# Pipeline Review: Churn Prediction Refresh
### AI-Generated Pipeline — Scheduled for Pre-Board Call Execution
This pipeline was generated by an AI agent overnight to refresh churn
predictions before the quarterly board call. Let's review the output
before it goes to production.

In [ ]:
-- AI-generated churn prediction refresh
-- Generated by: AI Pipeline Agent v2.1
-- Scheduled: Pre-board call refresh
SELECT 
    c.CUSTOMER_NAME,
    c.SEGMENT,
    c.CHURN_RISK_SCORE,
    COUNT(DISTINCT a.ACCOUNT_ID) AS account_count,
    SUM(a.MONTHLY_TRANSACTION_VOLUME) AS total_volume
FROM RETAILBANK_2028.PUBLIC.CUSTOMERS c
JOIN RETAILBANK_2028.PUBLIC.CUSTOMERS a
    ON c.CUSTOMER_ID = a.ACCOUNT_ID   -- BUG: AI chose wrong join key
WHERE c.CHURN_RISK_SCORE > 0.6
GROUP BY 1, 2, 3
ORDER BY c.CHURN_RISK_SCORE DESC;
-- Expected: should show at-risk customers for board presentation

## ⚠️ Result: ~1,847 at-risk customers
**Last month: 612 at-risk customers**

That's a **200% increase** in one month. Does that make business sense?

Let's investigate the join...

In [ ]:
-- How many unique customers vs accounts do we have?
SELECT 
    COUNT(DISTINCT CUSTOMER_ID) AS unique_customers,
    COUNT(DISTINCT ACCOUNT_ID) AS unique_accounts,
    COUNT(*) AS total_rows
FROM RETAILBANK_2028.PUBLIC.CUSTOMERS;
-- One customer can have MULTIPLE accounts
-- Joining customer_id = account_id creates a many-to-many explosion

In [ ]:
-- FIXED: No self-join needed — query directly from CUSTOMERS table
SELECT 
    CUSTOMER_NAME,
    SEGMENT,
    CHURN_RISK_SCORE,
    COUNT(DISTINCT ACCOUNT_ID) AS account_count,
    SUM(MONTHLY_TRANSACTION_VOLUME) AS total_volume
FROM RETAILBANK_2028.PUBLIC.CUSTOMERS
WHERE CHURN_RISK_SCORE > 0.6
GROUP BY 1, 2, 3
ORDER BY CHURN_RISK_SCORE DESC;
-- Result: ~589 at-risk customers — in line with last month's trend

## ✅ Fixed: ~589 at-risk customers
Back in line with the monthly trend.

### What happened?
The AI joined `CUSTOMER_ID = ACCOUNT_ID` — both are integers, so no
type error. But because one customer can have multiple accounts, this
created a **many-to-many relationship** that inflated results by ~3x.

### The lesson
The AI wrote 90% of this pipeline correctly. But it had no business
context. It didn't know that one customer can have multiple accounts.
**That knowledge lives in your head, not in the data.**

> Technical judgment is the skill. Knowing when the answer is wrong
> — even before you know why.